# Milvus の基本的な使い方

# 1、DDL 操作

## 1.1 データベース関連操作

### ① データベースを確認

例1：クライアントを操作

In [5]:

from pymilvus import MilvusClient

db_name = "rag_tutorial"
collection_name = "docs"

client = MilvusClient("http://localhost:19530")

例2：すべてのデータベースを一覧表示

In [6]:
existed_databases = client.list_databases()

for db in existed_databases:
    print(db)

default
rag_tutorial


### ② データベースを作成

In [7]:
if db_name not in existed_databases:
    client.create_database(db_name=db_name)

### ③ データベースを削除

データベース配下に Collection がある場合は削除できません。先にすべての Collection を削除してから Database を削除する必要があります

In [ ]:
client.drop_database(db_name=db_name)

## 1.2 Collection 関連操作

### ① データベースを切り替え

In [8]:
client.use_database(db_name=db_name)

### ② データベース配下の collections を確認

In [11]:
collections = client.list_collections()

for coll in collections:
    print(coll)

docs


### ③ collection を作成

In [ ]:
client.create_collection(
    collection_name=collection_name,
    dimension=3072,
    metric_type="COSINE"
)

### ④ collection を削除

In [ ]:
client.drop_collection(collection_name=collection_name)

# 2、DML 操作

## 2.1 埋め込みモデルの初期化

In [10]:
import os
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings

load_dotenv(override=True)

embed_model = init_embeddings(
    model="openai:text-embedding-3-large",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE"),
)

## 2.2 collection を準備

### ① collection を作成

In [ ]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=3072,
    metric_type="COSINE"
)

### ② collection のメタデータを確認

In [12]:
from rich import print as rprint

metadata = client.describe_collection(collection_name=collection_name)

rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 3072}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 468246679115556511,
    'consistency_level': 2,
    'properties': {},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 468248046354300938,
    'update_timestamp': 468248046354300938
}

## 2.3 データを準備

### ① 元データを準備

In [13]:
# テストデータを準備
texts = [
    "LangChain は LLM アプリケーションを構築するための開発フレームワークです。",
    "Milvus は AI アプリケーションに適したベクトルデータベースです。",
    "RAG の核心は、まず関連知識を検索し、その後大規模言語モデルに回答を生成させることです。",
    "Docker Desktop を使えば、ローカルで Milvus Standalone を簡単に実行できます。"
]

### ② 埋め込みベクトルを生成

In [14]:
vectors = embed_model.embed_documents(texts)

### ③ 生成された埋め込みベクトルを確認

In [15]:
print(len(vectors))

print(len(vectors[0]))

print(vectors[0][:5])

4
3072
[-0.038421630859375, -0.017822265625, -0.03131103515625, 0.0169525146484375, -0.0099945068359375]


### ④ 挿入可能なデータ形式にラップ

In [ ]:
data = [
    {
        "id": i,
        "vector": vectors[i],
        "text": texts[i],
        "source": "demo"
    } for i in range(len(texts))
]

## 2.4 データを書き込み

### ① データを挿入

In [ ]:
insert_res = client.upsert(
    collection_name=collection_name,
    data=data,
)

print("insert result : ", insert_res)

### ② 手動 flush

Milvus はすぐにはデータをディスクに書き込みません。書き込み結果を確認するために、手動で flush してデータをディスクに書き出します

In [17]:
client.flush(collection_name=collection_name)

### ③ collection の統計情報を確認

In [16]:
stats = client.get_collection_stats(collection_name=collection_name)

print("stats : ", stats)

stats :  {'row_count': 45}


# 3、DQL 操作

## 3.1 データをスキャン

In [ ]:

iterator = client.query_iterator(
    collection_name=collection_name,
    filter='text like "%言語モデル%"',
    output_fields=["*"]
)

i = 0
while True:

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"{i + 1} 件目のデータ：")
        # print(row)

        print(f"id : {row["id"]},vector = {row["vector"][:5]},text = {row["text"]},source = {row["source"]}")

        i += 1

iterator.close()

## 3.2 主キーでデータを照会

In [ ]:

res = client.get(
    collection_name=collection_name,
    ids=[0, 1, 2]
)

print(len(res))

for i in range(len(res)):
    print(f"{i + 1} 件目のデータ：")
    print(f"id : {res[i]["id"]},vector = {res[i]["vector"][:5]},text = {res[i]["text"]},source = {res[i]["source"]}")
    # print(res[i])


## 3.3 類似度検索

### ① クエリ埋め込みを準備

In [ ]:
# 類似度検索
query = "ベクトルデータベースとは何ですか？"
query_vector = embed_model.embed_query(query)

### ② 検索

In [ ]:
results = client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["text", "source", "id"]
)

for res in results[0]:
    print(res)